In [24]:
import pandas as pd

df = pd.read_csv("./data/gene expression/1_4_hpa_rna_celline.tsv", sep="\t")
print(df.shape)
print(df.dtypes)
print(df.head())

(24315372, 6)
Gene          object
Gene name     object
Cell line     object
TPM          float64
pTPM         float64
nTPM         float64
dtype: object
              Gene Gene name Cell line   TPM  pTPM  nTPM
0  ENSG00000000003    TSPAN6      143B  22.0  27.6  25.9
1  ENSG00000000003    TSPAN6     22Rv1   2.8   3.6   2.7
2  ENSG00000000003    TSPAN6  23132/87   6.2   7.5   7.5
3  ENSG00000000003    TSPAN6      253J  14.2  18.7  25.4
4  ENSG00000000003    TSPAN6   253J-BV  13.0  17.1  18.5


In [25]:
print(df.shape)          # rows × cols
print(df.isnull().sum())  # per-column NaN counts
print(df.dtypes)

(24315372, 6)
Gene         0
Gene name    0
Cell line    0
TPM          0
pTPM         0
nTPM         0
dtype: int64
Gene          object
Gene name     object
Cell line     object
TPM          float64
pTPM         float64
nTPM         float64
dtype: object


In [26]:
print(df.columns.tolist())
print(df.shape)
print(df.head(3))

['Gene', 'Gene name', 'Cell line', 'TPM', 'pTPM', 'nTPM']
(24315372, 6)
              Gene Gene name Cell line   TPM  pTPM  nTPM
0  ENSG00000000003    TSPAN6      143B  22.0  27.6  25.9
1  ENSG00000000003    TSPAN6     22Rv1   2.8   3.6   2.7
2  ENSG00000000003    TSPAN6  23132/87   6.2   7.5   7.5


In [27]:
print(df['Gene name'].nunique())   # gene symbols
print(df['Gene'].nunique())        # Ensembl IDs  
print(df['Cell line'].nunique())   # cell lines
print(df.isnull().sum())
print(df[['TPM','pTPM','nTPM']].describe())

20151
20162
1206
Gene         0
Gene name    0
Cell line    0
TPM          0
pTPM         0
nTPM         0
dtype: int64
                TPM          pTPM          nTPM
count  2.431537e+07  2.431537e+07  2.431537e+07
mean   3.894269e+01  4.955707e+01  5.081671e+01
std    3.840951e+02  4.894009e+02  6.013716e+02
min    0.000000e+00  0.000000e+00  0.000000e+00
25%    0.000000e+00  0.000000e+00  0.000000e+00
50%    2.800000e+00  3.600000e+00  3.600000e+00
75%    1.790000e+01  2.280000e+01  2.250000e+01
max    1.534615e+05  1.837973e+05  4.211731e+05


In [28]:
# look for columns like 'tpm', 'nx', 'ptpm', 'nTPM'
print(df.columns.tolist())
print(df.describe())

['Gene', 'Gene name', 'Cell line', 'TPM', 'pTPM', 'nTPM']
                TPM          pTPM          nTPM
count  2.431537e+07  2.431537e+07  2.431537e+07
mean   3.894269e+01  4.955707e+01  5.081671e+01
std    3.840951e+02  4.894009e+02  6.013716e+02
min    0.000000e+00  0.000000e+00  0.000000e+00
25%    0.000000e+00  0.000000e+00  0.000000e+00
50%    2.800000e+00  3.600000e+00  3.600000e+00
75%    1.790000e+01  2.280000e+01  2.250000e+01
max    1.534615e+05  1.837973e+05  4.211731e+05


In [29]:
# Get the unique cell line names
cell_lines = df['Cell line'].unique()
print(len(cell_lines))
print(sorted(cell_lines)[:30])   # eyeball for format

1206
['143B', '22Rv1', '23132/87', '253J', '253J-BV', '42-MG-BA', '537-mel', '5637', '59M', '624-mel', '639V', '647V', '697', '769-P', '786-O', '8-MG-BA', '8305C', '8505C', '888-mel', 'A-1207', 'A-172', 'A-204', 'A-253', 'A-375', 'A-427', 'A-431', 'A-498', 'A-549', 'A-673', 'A-704']


In [30]:
# If there's an ensembl column:
print(df['Gene'].isna().sum())
print(df['Gene'].str.startswith('ENSG').sum())

# If only symbol present, note this — will need HGNC mapping

0
24315372


In [31]:
df['cell_line_norm'] = df['Cell line'].str.upper().str.replace(r'[-\s]', '', regex=True).str.strip()

In [32]:
# Load sample_info for reference
sample_info = pd.read_csv("./data/nomenclature/9_DepMap_sample_info.csv")
depmap_names = set(sample_info['stripped_cell_line_name'].str.upper().str.replace(r'[-\s]', '', regex=True))
hpa_names = set(df['cell_line_norm'].unique())

overlap = depmap_names & hpa_names
only_depmap = depmap_names - hpa_names
only_hpa = hpa_names - depmap_names

print(f"Overlap: {len(overlap)}")
print(f"Only in DepMap: {len(only_depmap)}")
print(f"Only in HPA: {len(only_hpa)}")

Overlap: 1002
Only in DepMap: 838
Only in HPA: 204


In [33]:
# EGFR in A431
egfr_a431 = df[(df['Gene name'] == 'EGFR') & (df['Cell line'].str.upper().str.contains('A431'))]
print(egfr_a431)

# HER2 (ERBB2) in SKBR3
her2_skbr3 = df[(df['Gene name'].isin(['ERBB2', 'HER2'])) & (df['Cell line'].str.upper().str.contains('SKBR3'))]
print(her2_skbr3)

Empty DataFrame
Columns: [Gene, Gene name, Cell line, TPM, pTPM, nTPM, cell_line_norm]
Index: []
Empty DataFrame
Columns: [Gene, Gene name, Cell line, TPM, pTPM, nTPM, cell_line_norm]
Index: []


In [34]:
# 1. Check how many Ensembl IDs vs gene symbols differ (11 extra symbols vs IDs suggests aliases)
print(df[df['Gene name'] != df['Gene']].shape)  # should be all rows
# Check if any Gene IDs are non-ENSG format
non_ensg = df[~df['Gene'].str.startswith('ENSG')]
print(f"Non-ENSG rows: {len(non_ensg)}")
print(non_ensg['Gene'].unique()[:10])

# 2. Zero expression — what fraction?
zero_pct = (df['nTPM'] == 0).sum() / len(df) * 100
print(f"Zero nTPM: {zero_pct:.1f}%")

# 3. Check the 20,151 vs 20,162 discrepancy — Gene vs Gene name unique counts
# There are 11 more gene names than Ensembl IDs — likely some symbols map to same ENSG
dup_ensg = df.groupby('Gene')['Gene name'].nunique()
print(dup_ensg[dup_ensg > 1])  # Ensembl IDs with multiple symbols

# 4. Sanity check with HPA-style names (hyphenated)
egfr_a431 = df[(df['Gene name'] == 'EGFR') & (df['Cell line'] == 'A-431')]
print(egfr_a431)

her2_skbr3 = df[(df['Gene name'] == 'ERBB2') & (df['Cell line'].str.contains('SK-BR', case=False))]
print(her2_skbr3)

(23859504, 7)
Non-ENSG rows: 0
[]
Zero nTPM: 26.3%
Series([], Name: Gene name, dtype: int64)
                     Gene Gene name Cell line     TPM    pTPM    nTPM  \
10533229  ENSG00000146648      EGFR     A-431  2608.8  3187.9  2978.0   

         cell_line_norm  
10533229           A431  
                    Gene Gene name Cell line     TPM    pTPM    nTPM  \
9700822  ENSG00000141736     ERBB2   SK-BR-3  1638.4  1960.0  2448.4   

        cell_line_norm  
9700822          SKBR3  


In [35]:
# The 20,151 ENSG IDs in HPA vs ~19,200 protein-coding genes expected
# Check how many of HPA's genes are protein-coding by joining to HGNC
# For now, just check if pTPM == 0 where TPM > 0 — that flags non-protein-coding rows

non_coding_proxy = df[(df['TPM'] > 0) & (df['pTPM'] == 0)]
print(f"Rows with TPM>0 but pTPM=0: {len(non_coding_proxy)}")
print(f"Unique genes in that set: {non_coding_proxy['Gene'].nunique()}")

Rows with TPM>0 but pTPM=0: 0
Unique genes in that set: 0
